# SIGCTiArural — Benchmark V1 · Experimento M2_EfficientNetB0_001

## Resumen del experimento

**Objetivo:** comparar **EfficientNet-B0** contra el **baseline oficial M1 (MobileNetV2, CONGELADO)** bajo **exactamente las mismas condiciones experimentales** del Dataset V2+.

- **Dataset:** V2+ validado · 22.488 imágenes · 16 clases · 3 especies · split fijo train 15.741 / validation 3.373 / test 3.374 (seed 42).
- **Aceptación (idéntica a M1):** macro-F1 (validation) ≥ 0.955 · ECE ≤ 0.10 · diagnóstico sin fuga (< 0.90 → bug, ≈ 1.0 → fuga).
- **Comparabilidad:** mismas transformaciones, política de optimización, métricas, esquema de evaluación, definiciones de Macro-F1/ECE, matriz de confusión, reporte por clase, curvas y exportaciones.
- **Recordatorio gobernanza:** M1 NO se reabre, NO se reentrena. M1 = referencia de comparación.

**Resultado a determinar:** si EfficientNet-B0 supera o no a MobileNetV2 como **nuevo candidato principal** para SIGCTiArural V2+ (análisis automático al final del run).

## 1. Contexto y objetivo

| Atributo | Valor |
|---|---|
| experiment_id | `agriculture_v2_baseline_v2` |
| run | `M2_efficientnet_b0_001` |
| Baselines | **M1** MobileNetV2 · **MACRO-F1 val 0.9899 · ECE 0.0313** · 2.244.368 params (congelado) |
| Rol M2 (manifiesto §16) | `baseline_master_candidate` |
| Honestidad de estado | M2 = experimento a ejecutar; resultados `real` tras completarse. Test single-use al cierre. |

### Hipótesis de M2

| # | Hipótesis | Criterio de rechazo |
|---|---|---|
| H_M2.1 | Idéntica política → pipeline sano (sin fuga ni bug) | macro-F1 val < 0.90 o ≈ 1.0 |
| H_M2.2 | Calibración dentro de gate | ECE val > 0.10 |
| H_M2.3 (interés) | Como `baseline_master_candidate`, EfficientNet-B0 es **competitivo** con el control M1 (Δ macro-F1 ≥ −0.005) con **mejor o igual ECE** | Δ macro-F1 (M2 − M1) ≤ −0.005 (no reemplaza al control) |

> La decisión final del benchmark usará macro-F1 **sobre test (single-use)** + ECE + coste edge; M2 vs M1 sobre validation es **diagnóstico intermedio honesto**, no veredicto final.

## 2. Dataset V2+ (idéntico a M1)

- Particiones Tratamiento-Enfermedad de Plant-Village recuperado (22.488 representativas, 16 clases, 3 especies).
- **Split fijo (seed 42):** train 15.741 · validation 3.373 · test 3.374.
- Rutas esperadas bajo `curated/{train,validation,test}` con 16 carpetas de clase (mismas que M1).
- **Regla de oro:** el test set se toca **UNA sola vez** al cierre del run (celda protegida).

In [ ]:
# 0) Dependencias (idéntico a M1)
import importlib, subprocess, sys

for pkg in ["sklearn", "matplotlib", "PIL"]:
    try:
        importlib.import_module(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

import torch, torchvision, sklearn, numpy, PIL, matplotlib
print("torch", torch.__version__, "| torchvision", torchvision.__version__)
print("sklearn", sklearn.__version__, "| numpy", numpy.__version__, "| PIL", PIL.__version__, "| matplotlib", matplotlib.__version__)
print("CUDA disponible:", torch.cuda.is_available(), "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "sin GPU (CPU)")

In [ ]:
# 1) Semilla global + imports + dispositivo (AMPLIFICACIÓN: AMP migrado a torch.amp)
import os, random, math, time, csv, json, shutil, contextlib
import numpy as np
import torch, torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torchvision import datasets as tvd, transforms
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import (f1_score, balanced_accuracy_score,
                             precision_recall_fscore_support, confusion_matrix)
import matplotlib.pyplot as plt

SEED = 42
def set_seed(seed):
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
set_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Dispositivo:", DEVICE)
if DEVICE == "cpu":
    print("⚠️ Sin GPU: EfficientNet-B0 en CPU ≈ 10–14 h. Se recomienda Runtime → Cambiar tipo de ejecución → T4 GPU.")

# Corrección G1: AMP migrado de torch.cuda.amp a torch.amp (evita warnings de deprecación)
USE_AMP = DEVICE == "cuda"
if USE_AMP:
    from torch.amp import autocast, GradScaler
    def AMP_CTX():
        return autocast(device_type="cuda", dtype=torch.float16, enabled=True)
    def make_scaler():
        return GradScaler("cuda", enabled=True, init_scale=2**16)
else:
    def AMP_CTX():
        return contextlib.nullcontext()
    def make_scaler():
        return None

In [ ]:
# 2) Montar Google Drive y localizar el Dataset V2+ (curated) — CORRECCIÓN G2: salidas persistentes en Drive
from google.colab import drive
drive.mount("/content/drive")  # autorizar el permiso en el popup

DRIVE_ROOT = "/content/drive/MyDrive/SIGCTiArural"
DATA_ROOT = DRIVE_ROOT + "/datasets/v1/curated"
# Corrección G2: OUT dentro de Drive → los artefactos sobreviven al reinicio de runtime
OUT = DRIVE_ROOT + "/runs/M2_efficientnet_b0"
os.makedirs(OUT, exist_ok=True)

print("DATA_ROOT =", DATA_ROOT)
assert os.path.isdir(DATA_ROOT), f"No existe {DATA_ROOT}: verifica que el Dataset V2+ esté en Drive."
for p in ["train", "validation", "test"]:
    n_classes = len(os.listdir(os.path.join(DATA_ROOT, p)))
    print(f"{p}: {n_classes} carpetas de clase")
print("OUT (persistente en Drive) =", OUT)

In [ ]:
# 3) Reuso canónico: sincronizar benchmark/src (fuente única de métricas/datos) con fallback inline
# Preferimos importar los módulos oficiales de benchmark/ para NO duplicar código.
BM_SRC = None
for cand in [os.path.join(DRIVE_ROOT, "benchmark", "src"), "/content/benchmark/src"]:
    if os.path.isdir(cand):
        BM_SRC = cand
        break

if BM_SRC:
    DST = "/content/benchmark"
    if os.path.isdir(DST):
        shutil.rmtree(DST)
    shutil.copytree(BM_SRC, DST)
    if "/content" not in sys.path:
        sys.path.insert(0, "/content")
    import datasets as bm_datasets, metrics as bm_metrics, models as bm_models  # módulos canónicos
    print("Reuso: benchmark/src importado desde", BM_SRC)
    BM_REUSE = True
else:
    # FALLBACK: copia inline idéntica a benchmark/src (mismas fórmulas EXACTAS que M1)
    print("⚠️ benchmark/src no encontrado en Drive → usando copia inline canónica (idéntica a benchmark/src).")
    import numpy as _np, torch as _th
    def expected_calibration_error(conf, y_pred, y_true, n_bins=15):
        bins = _np.linspace(0, 1, n_bins + 1)
        total = _np.zeros(n_bins); correct = _np.zeros(n_bins)
        for c, p, t in zip(conf, y_pred, y_true):
            idx = min(int(_np.digitize(c, bins)) - 1, n_bins - 1); idx = max(idx, 0)
            total[idx] += 1; correct[idx] += int(p == t)
        mask = total > 0
        with _np.errstate(divide="ignore", invalid="ignore"):
            acc = _np.divide(correct, total, out=_np.zeros_like(total), where=mask)
        bin_conf = _np.zeros(n_bins); bin_conf[mask] = (bins[:-1][mask] + bins[1:][mask]) / 2
        return float(_np.sum((total / total.sum()) * _np.abs(acc - bin_conf)))
    def evaluate(probs, y_true, n_bins=15):
        probs = probs.cpu().numpy() if isinstance(probs, _th.Tensor) else _np.asarray(probs)
        y_true = y_true.cpu().numpy() if isinstance(y_true, _th.Tensor) else _np.asarray(y_true)
        y_pred = probs.argmax(axis=1); conf = probs.max(axis=1)
        macro_f1 = f1_score(y_true, y_pred, average="macro", zero_division=0)
        weighted_f1 = f1_score(y_true, y_pred, average="weighted", zero_division=0)
        bal_acc = balanced_accuracy_score(y_true, y_pred)
        p, r, f, _ = precision_recall_fscore_support(y_true, y_pred, zero_division=0)
        cm = confusion_matrix(y_true, y_pred)
        ece = expected_calibration_error(conf, y_pred, y_true, n_bins=n_bins)
        return {"macro_f1": float(macro_f1), "weighted_f1": float(weighted_f1),
                "balanced_accuracy": float(bal_acc), "per_class_precision": p.tolist(),
                "per_class_recall": r.tolist(), "per_class_f1": f.tolist(),
                "confusion_matrix": cm.tolist(), "ece": float(ece)}
    def class_weights(targets, num_classes):
        targets = _np.array(targets)
        counts = _np.bincount(targets, minlength=num_classes).astype(float)
        n = counts.sum()
        return _th.tensor(n / (counts * num_classes), dtype=_th.float32)
    from types import SimpleNamespace as _SNS
    bm_metrics = _SNS(evaluate=evaluate, expected_calibration_error=expected_calibration_error)
    bm_datasets = _SNS(class_weights=class_weights)
    bm_models = None
    BM_REUSE = False

In [ ]:
# 3b) Transformaciones + carga ImageFolder + verificación del split oficial (idéntico a M1)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def make_transforms(augment=True):
    if augment:
        return transforms.Compose([
            transforms.Resize(256),
            transforms.RandomResizedCrop(224),
            transforms.RandomHorizontalFlip(),
            transforms.RandomVerticalFlip(),
            transforms.RandomRotation(15),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            transforms.ToTensor(),
            transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
        ])
    return transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

train_ds = tvd.ImageFolder(os.path.join(DATA_ROOT, "train"), transform=make_transforms(augment=True))
val_ds   = tvd.ImageFolder(os.path.join(DATA_ROOT, "validation"), transform=make_transforms(augment=False))
test_ds  = tvd.ImageFolder(os.path.join(DATA_ROOT, "test"), transform=make_transforms(augment=False))

assert train_ds.classes == val_ds.classes == test_ds.classes, "Las particiones tienen distintas listas de clases"
CLASS_NAMES = train_ds.classes
N_CLASSES = len(CLASS_NAMES)
print("Clases:", N_CLASSES, "|", CLASS_NAMES)
print(f"Conteos verificados -> train {len(train_ds)} · val {len(val_ds)} · test {len(test_ds)} · total {len(train_ds)+len(val_ds)+len(test_ds)}")
assert N_CLASSES == 16 and len(train_ds) == 15741 and len(val_ds) == 3373 and len(test_ds) == 3374, "Los conteos no coinciden con el split oficial" 

## 3. EDA — Análisis Exploratorio (mismo esquema que M1)

Verificación de distribuciones y dataset card sobre las particiones oficiales (solo conteos; test intacto).

In [ ]:
# EDA) Distribución de clases por partición + especies + heatmap + dataset card visual (idéntico a M1)
import matplotlib as mpl

def _pcc(ds):
    return np.bincount(np.array(ds.targets), minlength=N_CLASSES)

c_tr = _pcc(train_ds); c_va = _pcc(val_ds); c_te = _pcc(test_ds)
tot = c_tr + c_va + c_te
species = {}
for i, name in enumerate(CLASS_NAMES):
    sp = name.split("__")[0]
    species.setdefault(sp, {"classes": 0, "images": 0})
    species[sp]["classes"] += 1
    species[sp]["images"] += int(tot[i])

print("=== Dataset Card (resumen visual) ===")
print(f"Dataset V2+ Oficial · {int(tot.sum())} imágenes · {N_CLASSES} clases · {len(species)} especies")
for sp in sorted(species):
    nc = species[sp]["classes"]; ni = species[sp]["images"]
    print(f"  {sp:<8} {nc:>2} clases · {ni:>6,} imágenes")
print(f"Ratio min/max: {int(tot.min())} -> {int(tot.max())} = {tot.max()/tot.min():.2f}×")
print(f"  min={CLASS_NAMES[int(tot.argmin())]} · max={CLASS_NAMES[int(tot.argmax())]}")
print("Suma verificada: " + str(int(tot.sum())) + " == 22488 -> " + ("OK" if int(tot.sum())==22488 else "FALLA"))

fig, ax = plt.subplots(figsize=(14, 5))
x = np.arange(N_CLASSES)
ax.bar(x, c_tr, label="train", color="#4C72B0")
ax.bar(x, c_va, bottom=c_tr, label="val", color="#DD8452")
ax.bar(x, c_te, bottom=c_tr + c_va, label="test", color="#55A868")
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=8)
ax.set_title("Distribución por clase × partición (Dataset V2+ oficial, 22.488)")
ax.legend(); plt.tight_layout(); plt.show()

In [ ]:
# EDA: Samplers y pesos por clase (misma fórmula canónica que M1: W = n/(n_c·C))
def worker_init_fn(seed, worker_id):
    np.random.seed(seed + worker_id); random.seed(seed + worker_id)

BATCH = 64
train_targets = np.array(train_ds.targets)
class_counts = np.bincount(train_targets)
w = 1.0 / (class_counts[train_targets].astype(float))
sampler = WeightedRandomSampler(w, num_samples=len(train_targets), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=2,
                          pin_memory=True, worker_init_fn=lambda wid: worker_init_fn(SEED, wid))
val_loader   = DataLoader(val_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds, batch_size=BATCH, shuffle=False, num_workers=2, pin_memory=True)

weights = bm_datasets.class_weights(train_ds.targets, N_CLASSES)
print("Pesos CE (min→max):", float(weights.min()), "→", float(weights.max()))

## 4. Ficha técnica del modelo — EfficientNet-B0

En la celda siguiente se miden (no se estiman a ojo) **parámetros**, **FLOPs/MACs aproximados**, **tamaño en memoria** y consumo estimado para **edge deployment**.

In [ ]:
# 5) Modelo EfficientNet-B0 (torchvision, pretrained) + cabeza de 16 clases + ficha técnica
from torchvision import models as tv_models

model = tv_models.efficientnet_b0(weights=tv_models.EfficientNet_B0_Weights.IMAGENET1K_V1)
model.classifier[1] = nn.Linear(model.classifier[1].in_features, N_CLASSES)
model = model.to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
n_train = sum(p.numel() for p in model.parameters() if p.requires_grad)

def count_macs(model, input_size=(1, 3, 224, 224)):
    counter = {"macs": 0}
    hooks = []
    def _hook(m, _inp, out):
        if isinstance(m, nn.Conv2d):
            n, oc, oh, ow = out.shape
            kh, kw = m.kernel_size
            # groups: depthwise (groups=in_channels) y pointwise (groups=1): MACs por salida = kh*kw*(in_channels/groups)
            cin_eff = m.in_channels // m.groups
            counter["macs"] += n * oc * oh * ow * (kh * kw * cin_eff)
        elif isinstance(m, nn.Linear):
            counter["macs"] += out.shape[0] * m.in_features * m.out_features
    def _reg(m):
        if isinstance(m, (nn.Conv2d, nn.Linear)):
            hooks.append(m.register_forward_hook(_hook))
    model.apply(_reg)
    was_training = model.training
    model.eval()
    with torch.no_grad(), AMP_CTX():
        _ = model(torch.randn(*input_size, device=DEVICE))
    model.train(was_training)
    for h in hooks:
        h.remove()
    return counter["macs"]

# Evita descargar pesos dos veces y mide en un solo forward ligero
macs = count_macs(model)
flops = 2 * macs
size_fp32_mb = n_params * 4 / 1e6
size_fp16_mb = n_params * 2 / 1e6
size_int8_kb = n_params / 1e3  # ≈ KB de pesos TFLite int8

print("=== FICHA TÉCNICA EfficientNet-B0 (medida) ===")
print(f"Parámetros totales : {n_params:,}  ({n_params/1e6:.2f}M) · trainables {n_train:,}")
print(f"MACs @224x224     : {macs/1e6:.1f}M  (FLOPs ≈ {flops/1e6:.1f}M)")
print(f"Tamaño checkpoint : {size_fp32_mb:.1f} MB (fp32) · ~{size_fp16_mb:.1f} MB (fp16) · ~{size_int8_kb:.0f} KB (int8 TFLite)")
print(f"Edge estimado     : pico de memoria de inferencia ≈ {size_int8_kb:.0f} KB solo pesos (int8); "
      f"flotante de propósito general T4/Colab sin problema")

FICHA = {
    "arquitectura": "EfficientNet-B0", "run": "M2_efficientnet_b0_001",
    "n_params": int(n_params), "n_trainable": int(n_train),
    "macs": int(macs), "flops_aprox": int(flops),
    "size_fp32_mb": round(size_fp32_mb, 2), "size_fp16_mb": round(size_fp16_mb, 2),
    "size_int8_kb": round(size_int8_kb, 0), "input": "224x224x3",
}
with open(os.path.join(OUT, "config.json"), "w") as f:
    json.dump(FICHA, f, indent=2, ensure_ascii=False)

## 5. Configuración canónica y comparabilidad

**Mismas condiciones que M1 (política idéntica):**

| Parámetro | Valor |
|---|---|
| batch_size | 64 |
| epochs | 40 (early stop patience 8, macro-F1 val) |
| optimizer | AdamW · lr 3e-4 · weight_decay 1e-4 |
| scheduler | CosineAnnealingLR (T_max=40, eta_min=lr·0.01) |
| loss | CrossEntropyLoss ponderada (w_c = N/(n_c·16)) |
| sampler | WeightedRandomSampler (balanceo por clase) |
| image_size | 224 (Resize 256 → crop) · seed 42 |
| transfer | pretrained ImageNet |

**Única diferencia (correcciones G aplicadas, sin efecto en métricas):** AMP en `torch.amp` · salidas persistentes en Drive · guardado automático por epoch.

In [ ]:
# 6) Pérdida, optimizador, schedule, AMP (migrado a torch.amp) — config canónica idéntica a M1
criterion = nn.CrossEntropyLoss(weight=weights.to(DEVICE))
optimizer = AdamW(model.parameters(), lr=3e-4, weight_decay=1e-4)
scheduler = CosineAnnealingLR(optimizer, T_max=40, eta_min=3e-4 * 0.01)
scaler = make_scaler()

print("EfficientNet-B0 | CE ponderada | AdamW 3e-4 wd 1e-4 | CosineAnnealing T=40 | AMP (torch.amp)")

## 6. Entrenamiento y evaluación (mismas métricas que M1)

In [ ]:
# 7) Ciclo de entrenamiento por epoch (fiel a benchmark/src/train.py; AMP migrado a torch.amp)
def run_one_epoch(model, loader, criterion, optimizer=None, scaler=None):
    is_train = optimizer is not None
    model.train(is_train)
    total_loss, correct, n = 0.0, 0.0, 0
    all_probs, all_targets = [], []
    with torch.set_grad_enabled(is_train):
        for images, targets in loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            with AMP_CTX():
                logits = model(images)
                loss = criterion(logits, targets)
            if is_train:
                optimizer.zero_grad()
                if scaler is not None:
                    scaler.scale(loss).backward()
                    scaler.step(optimizer); scaler.update()
                else:
                    loss.backward(); optimizer.step()
            probs = torch.softmax(logits.detach(), dim=1)
            all_probs.append(probs); all_targets.append(targets)
            total_loss += loss.item() * images.size(0)
            correct += (probs.argmax(1) == targets).sum().item(); n += images.size(0)
    m = bm_metrics.evaluate(torch.cat(all_probs), torch.cat(all_targets))
    m["loss"] = total_loss / n; m["accuracy"] = correct / n
    return m

In [ ]:
# 8) LOOP DE ENTRENAMIENTO (40 epochs · patience 8) — CORRECCIÓN G3: guardado automático en Drive
EPOCHS = 40; PATIENCE = 8
os.makedirs(os.path.join(OUT, "checkpoints"), exist_ok=True)
os.makedirs(os.path.join(OUT, "plots"), exist_ok=True)
os.makedirs(os.path.join(OUT, "report"), exist_ok=True)

best_f1, best_epoch, no_improve = -1.0, -1, 0
rows = []
t_start = time.time()

hdr = ["epoch", "train_loss", "train_macro_f1", "val_loss", "val_macro_f1", "val_ece", "lr", "time_s"]
# CORRECCIÓN G3: CSV incremental con flush por epoch (reanudable ante reinicio de runtime)
csv_path = os.path.join(OUT, "metrics.csv")
write_header = (not os.path.exists(csv_path)) or os.path.getsize(csv_path) == 0
with open(csv_path, "a", newline="") as f:
    fcsv = csv.writer(f)
    if write_header:
        fcsv.writerow(hdr)
        f.flush()
    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        tr = run_one_epoch(model, train_loader, criterion, optimizer, scaler)
        va = run_one_epoch(model, val_loader, criterion, scaler=scaler)
        scheduler.step()
        lr = optimizer.param_groups[0]["lr"]
        row = [epoch, round(tr["loss"],4), round(tr["macro_f1"],4),
               round(va["loss"],4), round(va["macro_f1"],4),
               round(va["ece"],4), round(lr,6), round(time.time()-t0,1)]
        rows.append([epoch, tr["loss"], tr["macro_f1"], va["loss"], va["macro_f1"], va["ece"], lr, time.time()-t0])
        fcsv.writerow(row); f.flush()
        print(f"epoch={epoch} train_f1={tr['macro_f1']:.4f} val_f1={va['macro_f1']:.4f} "
              f"val_ece={va['ece']:.4f} lr={lr:.2e} ({time.time()-t0:.0f}s)", flush=True)

        # Guardado automático (siempre en Drive): best solo en mejora, last por epoch
        improved = va["macro_f1"] > best_f1
        if improved:
            best_f1, best_epoch, no_improve = va["macro_f1"], epoch, 0
            torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "macro_f1_val": best_f1, "ece_val": va["ece"]},
                       os.path.join(OUT, "checkpoints", "best.pth"))
        else:
            no_improve += 1
        torch.save({"epoch": epoch, "model_state_dict": model.state_dict(),
                    "macro_f1_val": va["macro_f1"]}, os.path.join(OUT, "checkpoints", "last.pth"))
        if no_improve >= PATIENCE:
            print(f"Early stop en epoch {epoch} (best {best_f1:.4f} @ {best_epoch})")
            break

print(f"\nFIN: best_val_macro_f1={best_f1:.4f} @ epoch {best_epoch} · "
      f"duración total {(time.time()-t_start)/3600:.2f} h")

In [ ]:
# 8b) Ficha post-entrenamiento (tiempo por época) — mejora académica
rows_arr = np.array(rows)
total_s = rows_arr[:, 7].sum()
mean_epoch_s = rows_arr[:, 7].mean()
best_epoch_run = [int(e) for e in rows_arr[:, 0]][int(np.argmax(rows_arr[:, 4]))]

FICHA["total_time_s"] = round(float(total_s), 1)
FICHA["mean_time_epoch_s"] = round(float(mean_epoch_s), 1)
FICHA["best_epoch"] = best_epoch_run
FICHA["best_val_macro_f1"] = float(rows_arr[rows_arr[:, 0].astype(int) == best_epoch_run, 4][0])
with open(os.path.join(OUT, "config.json"), "w", encoding="utf-8") as f:
    json.dump(FICHA, f, indent=2, ensure_ascii=False)

print(f"Tiempo por época promedio: {mean_epoch_s:.0f}s · total: {total_s/3600:.2f} h")
print(f"Mejor val macro-F1 en las métricas históricas: epoch {best_epoch_run} · {FICHA['best_val_macro_f1']:.4f}")

In [ ]:
# 9) Evaluación final sobre VALIDATION con el checkpoint best (idéntico a M1) → exportación automática
state = torch.load(os.path.join(OUT, "checkpoints", "best.pth"), map_location=DEVICE, weights_only=False)
model.load_state_dict(state["model_state_dict"])
model.eval()

probs_t, y_t = [], []
with torch.no_grad():
    for images, targets in val_loader:
        images, targets = images.to(DEVICE), targets.to(DEVICE)
        with torch.no_grad(), AMP_CTX():
            logits = model(images)
        probs_t.append(torch.softmax(logits, dim=1).cpu()); y_t.append(targets.cpu())
res = bm_metrics.evaluate(torch.cat(probs_t), torch.cat(y_t))

print(f"=== Resultado sobre VALIDATION (best @ epoch {state['epoch']}) ===")
print(f"macro_f1 = {res['macro_f1']:.4f}  (gate ≥ 0.955)")
print(f"ece       = {res['ece']:.4f}       (gate ≤ 0.10)")
print(f"bal_acc   = {res['balanced_accuracy']:.4f}")
print(f"weighted_f1 = {res['weighted_f1']:.4f}")

# Exportación automática ≥ checkpoint de validation (Corrección: persistente en Drive, no efímero)
with open(os.path.join(OUT, "report", "validation_metrics.json"), "w", encoding="utf-8") as f:
    json.dump(res, f, indent=2, ensure_ascii=False)
print("Guardado:", os.path.join(OUT, "report", "validation_metrics.json"))

In [ ]:
# 10) Curvas train/val + matriz de confusión + calibración (idéntico a M1) → PNG en Drive
rows_a = np.array(rows)
ep = rows_a[:, 0].astype(int)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
axes[0].plot(ep, rows_a[:, 1], label="train"); axes[0].plot(ep, rows_a[:, 3], label="val")
axes[0].set_title("Loss"); axes[0].legend()
axes[1].plot(ep, rows_a[:, 2], label="train"); axes[1].plot(ep, rows_a[:, 4], label="val")
axes[1].axhline(0.955, ls="--", c="r", label="gate 0.955"); axes[1].set_title("macro-F1"); axes[1].legend()
axes[2].plot(ep, rows_a[:, 5])
axes[2].axhline(0.10, ls="--", c="r", label="gate 0.10"); axes[2].set_title("ECE val"); axes[2].legend()
plt.tight_layout(); plt.savefig(os.path.join(OUT, "plots", "train_val_curves.png"), dpi=120)
plt.show()

cm = np.array(res["confusion_matrix"])
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(cm / np.maximum(cm.sum(1, keepdims=True), 1), cmap="Blues")
ax.set_xticks(range(N_CLASSES)); ax.set_yticks(range(N_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=90); ax.set_yticklabels(CLASS_NAMES)
plt.colorbar(im); plt.title("Confusion Matrix (validation, normalizada por fila) — M2")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "plots", "confusion_matrix.png"), dpi=120)
plt.show()

In [ ]:
# 11) ROC (one-vs-rest) + Precision-Recall per class + Calibration Curve (validation) — M2
from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

probs_v = torch.cat(probs_t).numpy()
y_v = torch.cat(y_t).numpy()
y_ohe = np.eye(N_CLASSES)[y_v]
y_pred_v = probs_v.argmax(1)

fig, ax = plt.subplots(figsize=(8, 8))
fpr_m, tpr_m, _ = roc_curve(y_ohe.ravel(), probs_v.ravel())
roc_micro = auc(fpr_m, tpr_m)
roc_curves = [roc_curve(y_ohe[:, k], probs_v[:, k]) for k in range(N_CLASSES)]
union_fpr = np.unique(np.concatenate([f for f, _, _ in roc_curves]))
mean_tpr = np.mean([np.interp(union_fpr, f, t) for f, t, _ in roc_curves], axis=0)
roc_macro = auc(union_fpr, mean_tpr)
for fpr_k, tpr_k, _ in roc_curves:
    ax.plot(fpr_k, tpr_k, alpha=0.25)
ax.plot(fpr_m, tpr_m, lw=2, color="navy", label=f"ROC micro (AUC={roc_micro:.3f})")
ax.plot([0, 1], [0, 1], "k--")
ax.set_xlabel("FPR"); ax.set_ylabel("TPR")
ax.set_title(f"ROC one-vs-rest — M2 (macro-AUC={roc_macro:.3f})")
ax.legend(); plt.tight_layout(); plt.savefig(os.path.join(OUT, "plots", "roc_1vRest.png"), dpi=120); plt.show()

fig, ax = plt.subplots(figsize=(8, 8))
aps = []
for k in range(N_CLASSES):
    prec_k, rec_k, _ = precision_recall_curve(y_ohe[:, k], probs_v[:, k])
    aps.append(average_precision_score(y_ohe[:, k], probs_v[:, k]))
    ax.plot(rec_k, prec_k, alpha=0.3)
ax.plot([0, 1], [0, 1], "k--", alpha=0.3)
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_title(f"PR per class — M2 (mAP={np.mean(aps):.3f})")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "plots", "pr_per_class.png"), dpi=120); plt.show()

conf_v = probs_v.max(1)
NB = 15; bins = np.linspace(0, 1, NB + 1)
b_mean = np.zeros(NB); b_acc = np.zeros(NB); b_cnt = np.zeros(NB)
for c, yp, yt in zip(conf_v, y_pred_v, y_v):
    b = min(int(np.digitize(c, bins)) - 1, NB - 1); b = max(b, 0)
    b_mean[b] += c; b_acc[b] += int(yp == yt); b_cnt[b] += 1
mask = b_cnt > 0
b_mean[mask] = b_mean[mask] / b_cnt[mask]
b_acc[mask] = b_acc[mask] / b_cnt[mask]
fig, ax = plt.subplots(figsize=(7, 7))
ax.plot([0, 1], [0, 1], "k--", label="perfecta")
ax.plot(b_mean[mask], b_acc[mask], "o-", color="#DD8452", label=f"modelo M2 (ECE={res['ece']:.3f})")
ax.set_xlabel("Confianza predicha"); ax.set_ylabel("Precisión observada")
ax.set_title("Calibration Curve (reliability diagram) — validation M2")
ax.legend(); plt.tight_layout(); plt.savefig(os.path.join(OUT, "plots", "calibration.png"), dpi=120); plt.show()

In [ ]:
# 12) Reporte per-class + gate final (idéntico a M1)
print(f"{'Clase':<42} {'P':>6} {'R':>6} {'F1':>6}")
for i, name in enumerate(CLASS_NAMES):
    print(f"{name:<42} {res['per_class_precision'][i]:>6.3f} {res['per_class_recall'][i]:>6.3f} {res['per_class_f1'][i]:>6.3f}")

gate_f1 = res["macro_f1"] >= 0.955
gate_ece = res["ece"] <= 0.10
print("\n=== GATE FINAL (M2) ===")
print(f"macro_f1_val {res['macro_f1']:.4f} ≥ 0.955 : {'PASS ✅' if gate_f1 else 'FAIL ❌'}")
print(f"ece_val      {res['ece']:.4f} ≤ 0.10  : {'PASS ✅' if gate_ece else 'FAIL ❌'}")

if res["macro_f1"] < 0.90:
    print("\n⚠️ macro_f1_val < 0.90 → sospechar bug de balanceo/labels. NO iterar contra test; revisar pipeline.")
elif gate_f1 and gate_ece:
    print("\n✅ M2 supera gates. Siguiente paso: análisis M2 vs M1 y evaluar TEST una sola vez.")
else:
    print("\n⚠️ M2 dentro de rango diagnóstico pero no alcanza gates. Registrar ECE/F1 y comparar con arquitecturas.")

In [ ]:
# 12b) Error Analysis sobre validation (idéntico a M1) → PNG en Drive
cm_e = np.array(res["confusion_matrix"])
order = np.argsort(res["per_class_f1"])
print("=== Clases con peor F1 (validation) ===")
for i in order[:5]:
    fi = res["per_class_f1"][i]; pi = res["per_class_precision"][i]; ri = res["per_class_recall"][i]
    sop = int(cm_e[i].sum())
    print(f"  {CLASS_NAMES[i]:<42} F1={fi:.3f} P={pi:.3f} R={ri:.3f} soporte={sop}")

print("\n=== Parejas de confusión más frecuentes (off-diagonal, validation) ===")
cm_off = cm_e.copy(); np.fill_diagonal(cm_off, 0)
pairs = []
for i in range(N_CLASSES):
    for j in range(N_CLASSES):
        if i != j and cm_off[i, j] > 0:
            pairs.append((cm_off[i, j], i, j))
for n, i, j in sorted(pairs, reverse=True)[:8]:
    print(f"  {CLASS_NAMES[i]} -> {CLASS_NAMES[j]} : {int(n)} casos")

fig, ax = plt.subplots(figsize=(11, 9))
norm_e = cm_e / np.maximum(cm_e.sum(1, keepdims=True), 1)
np.fill_diagonal(norm_e, np.nan)
im = ax.imshow(norm_e, cmap="Reds")
ax.set_xticks(range(N_CLASSES)); ax.set_yticks(range(N_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=90); ax.set_yticklabels(CLASS_NAMES)
plt.colorbar(im, label="% de la fila (errores)")
plt.title("Error Analysis — M2 (validation, sin diagonal)")
plt.tight_layout(); plt.savefig(os.path.join(OUT, "plots", "error_analysis.png"), dpi=120); plt.show()

## 7. Análisis final automático: "M2 vs Baseline M1"

Comparación (validation) contra el baseline **congelado** M1 (MobileNetV2 · macro-F1 0.9899 · ECE 0.0313 · 2.244.368 params). Δ con signo **M2 − M1**.

In [ ]:
# 13) Análisis automático M2 vs Baseline M1 (+ exportación + recomendación SIGCTiArural)
# --- Baseline M1 (registro oficial del dossier; NO se modifica) ---
M1 = {
    "arquitectura": "MobileNetV2", "macro_f1": 0.9899, "ece": 0.0313,
    "balanced_accuracy": 0.9898, "weighted_f1": 0.9902,
    "n_params": 2244368,
    "time_h": None,  # tiempo total de M1 NO declarado en el dossier (honestidad)
}

M2 = {
    "arquitectura": "EfficientNet-B0", "macro_f1": res["macro_f1"], "ece": res["ece"],
    "balanced_accuracy": res["balanced_accuracy"], "weighted_f1": res["weighted_f1"],
    "n_params": int(n_params),
    "time_h": round(float(total_s) / 3600, 2),
    "time_epoch_s": round(float(mean_epoch_s), 1),
}

def diff(a, b):
    return None if (a is None or b is None) else round(a - b, 4)

d = {
    "macro_f1": round(M2["macro_f1"] - M1["macro_f1"], 4),
    "ece": round(M2["ece"] - M1["ece"], 4),
    "balanced_accuracy": round(M2["balanced_accuracy"] - M1["balanced_accuracy"], 4),
    "weighted_f1": round(M2["weighted_f1"] - M1["weighted_f1"], 4),
    "params_pct": round((M2["n_params"] - M1["n_params"]) / M1["n_params"] * 100, 1),
    "time_h": diff(M2["time_h"], M1["time_h"]),
}

print("=== M2 (EfficientNet-B0) vs BASELINE M1 (MobileNetV2) — validation ===")
print(f"Macro-F1      : M1={M1['macro_f1']:.4f} · M2={M2['macro_f1']:.4f} · Δ={d['macro_f1']:+.4f}")
print(f"ECE           : M1={M1['ece']:.4f} · M2={M2['ece']:.4f} · Δ={d['ece']:+.4f}")
print(f"Balanced Acc  : M1={M1['balanced_accuracy']:.4f} · M2={M2['balanced_accuracy']:.4f} · Δ={d['balanced_accuracy']:+.4f}")
print(f"Weighted F1   : M1={M1['weighted_f1']:.4f} · M2={M2['weighted_f1']:.4f} · Δ={d['weighted_f1']:+.4f}")
print(f"Parámetros    : M1={M1['n_params']:,} · M2={M2['n_params']:,} · Δ={d['params_pct']:+.1f}%")
print(f"Tiempo total  : M1={'N/D (no declarado)' if M1['time_h'] is None else M1['time_h']} h · "
      f"M2={M2['time_h']} h · M2 por época ≈ {M2['time_epoch_s']}s")

# --- Lógica de recomendación (criterio benchmark: ECE → macro-F1 → coste edge) ---
supera = (d["macro_f1"] >= -0.005) and (d["ece"] <= 0.010)
if supera:
    rec = "M2 (EfficientNet-B0) es COMPETITIVO: no degrada macro-F1 (Δ ≥ −0.005) y mantiene calibración comparable. "           "Se mantiene como baseline_master_candidate. La decisión final depende del test single-use (single-level)."
else:
    rec = "M2 NO supera al control en validation (degradación de macro-F1 > 0.005 o ECE peor). "           "MobileNetV2 sigue siendo referente; valorar ECE/coste edge antes del test final."
if M2["ece"] > 0.10 or M2["macro_f1"] < 0.955:
    rec += " (OJO: M2 no cumpliría gates de aceptación sobre validation.)"

print("\n=== RECOMENDACIÓN SIGCTiArural ===")
print(rec)

analysis = {"M1": M1, "M2": M2, "diff": d, "recomendacion": rec,
            "nota": "Comparación sobre VALIDATION (diagnóstico honesto). El veredicto final usa test single-use + ECE + coste edge."}
with open(os.path.join(OUT, "report", "M2_VS_M1.json"), "w", encoding="utf-8") as f:
    json.dump(analysis, f, indent=2, ensure_ascii=False)

# Tabla comparativa exportada (para GitHub/defensa)
tabla = [
    ["Modelo", "Macro-F1", "ECE", "BalAcc", "W-F1", "Params (M)", "Tiempo total (h)"],
    ["M1 MobileNetV2", format(M1["macro_f1"], ".4f"), format(M1["ece"], ".4f"),
     format(M1["balanced_accuracy"], ".4f"), format(M1["weighted_f1"], ".4f"),
     f"{M1['n_params']/1e6:.2f}", "N/D" if M1["time_h"] is None else str(M1["time_h"])],
    ["M2 EfficientNet-B0", format(M2["macro_f1"], ".4f"), format(M2["ece"], ".4f"),
     format(M2["balanced_accuracy"], ".4f"), format(M2["weighted_f1"], ".4f"),
     f"{M2['n_params']/1e6:.2f}", str(M2["time_h"])],
]
print("\n--- TABA COMPARATIVA (M1 vs M2) ---")
for r in tabla:
    print(" | ".join(r))
md_lines = "\n".join(["| " + " | ".join(r) + " |" for r in tabla])
with open(os.path.join(OUT, "report", "M2_VS_M1.md"), "w", encoding="utf-8") as f:
    f.write("# M2 vs Baseline M1 (SIGCTiArural)\n\n" + md_lines + "\n\n" + rec + "\n")

## 8. Uso del TEST (regla de oro — UNA sola vez al cierre)

Ejecutar **solo cuando M2 esté aprobado** sobre validation y se haya decidido publicar como resultado final.

In [ ]:
# 14) [SINGLE-USE] Evaluación final sobre TEST — ejecutar UNA sola vez al cierre del run
RUN_FINAL_TEST = False   # ← cambiar a True SOLO al cierre tras aprobar validation

if RUN_FINAL_TEST:
    model.load_state_dict(torch.load(os.path.join(OUT, "checkpoints", "best.pth"),
                                     map_location=DEVICE, weights_only=False)["model_state_dict"])
    model.eval()
    probs_t, y_t = [], []
    with torch.no_grad():
        for images, targets in test_loader:
            images, targets = images.to(DEVICE), targets.to(DEVICE)
            with torch.no_grad(), AMP_CTX():
                logits = model(images)
            probs_t.append(torch.softmax(logits, dim=1).cpu()); y_t.append(targets.cpu())
    result_test = bm_metrics.evaluate(torch.cat(probs_t), torch.cat(y_t))
    with open(os.path.join(OUT, "report", "test_metrics.json"), "w", encoding="utf-8") as f:
        json.dump(result_test, f, indent=2, ensure_ascii=False)
    print("RESULTADO SOBRE TEST (referencia final single-use):")
    print(f"  macro_f1={result_test['macro_f1']:.4f} · weighted_f1={result_test['weighted_f1']:.4f} "
          f"· bal_acc={result_test['balanced_accuracy']:.4f} · ECE={result_test['ece']:.4f}")
    print("Guardado en", os.path.join(OUT, "report", "test_metrics.json"))
else:
    print("Celda protegida: RUN_FINAL_TEST=False. Cambiar a True SOLO al cierre del run tras aprobar validation.")

## 9. Salidas y trazabilidad

**Artefactos persistentes (Drive, sobreviven reinicios de Colab):**

```
{OUT}/
├── config.json                      # ficha técnica (params, FLOPs, tamaño, tiempos)
├── metrics.csv                      # epoch-by-epoch (flush por epoch)
├── checkpoints/best.pth  last.pth   # guardado automático por epoch
├── plots/
│   ├── train_val_curves.png
│   ├── confusion_matrix.png
│   ├── roc_1vRest.png
│   ├── pr_per_class.png
│   ├── calibration.png
│   └── error_analysis.png
└── report/
    ├── validation_metrics.json     # métricas canónicas (macro-F1, ECE, per-class, CM)
    ├── M2_VS_M1.json + M2_VS_M1.md # análisis M2 vs baseline
    └── test_metrics.json           # SOLO tras el single-use
```

**Reglas de gobernanza:**

1. M1 congelado — no se reentrena, no se modifica.
2. Dataset V2+ intocado (splits y manifiestos frozen).
3. Veredicto final: single-use sobre test + ECE + coste edge.
4. Reproducibilidad: política idéntica a M1 (seed 42, transformaciones, esquema de evaluación).